# 🎛️ EnerGIS Interactive Dashboard

Interaktives Dashboard für die Analyse von EnerGIS Optimierungsergebnissen.

## Features
- 📊 **Overview**: KPIs und Zusammenfassung auf einen Blick
- 📈 **Zeitreihen**: Interaktive Plots mit Komponenten-Auswahl und Zoom
- 💰 **Kosten**: Detaillierte Kostenanalyse mit interaktiven Tabellen
- 🏭 **Anlagen-Design**: Kapazitäten und Auslegung
- 🔀 **Vergleich**: PF vs RH/MPC Vergleich

## Verwendung
1. Führe die Setup-Zellen aus
2. Starte die Optimierung
3. Erstelle das Dashboard
4. Interagiere mit den Plots und Tabellen

---

## 1. Setup & Dependencies

In [ ]:
# Auto-Setup: Projekt-Root finden
from pathlib import Path
import sys
import os

def find_project_root(start: Path) -> Path:
    for candidate in [start] + list(start.parents):
        if (candidate / '.git').exists() and (candidate / 'energis').exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"✅ Projekt-Root: {PROJECT_ROOT}")

In [ ]:
# Prüfe Dependencies
missing_deps = []

try:
    import panel as pn
    print(f"✅ Panel {pn.__version__} verfügbar")
except ImportError:
    print("❌ Panel nicht gefunden")
    missing_deps.append("panel")

try:
    import holoviews as hv
    print(f"✅ Holoviews {hv.__version__} verfügbar")
except ImportError:
    print("❌ Holoviews nicht gefunden")
    missing_deps.append("holoviews")

try:
    import plotly
    print(f"✅ Plotly {plotly.__version__} verfügbar")
except ImportError:
    print("❌ Plotly nicht gefunden")
    missing_deps.append("plotly")

if missing_deps:
    print(f"\n⚠️  Fehlende Dependencies: {', '.join(missing_deps)}")
    print(f"\n📦 Installation:")
    print(f"   pip install {' '.join(missing_deps)} bokeh")
else:
    print("\n✅ Alle Dependencies verfügbar!")

In [ ]:
# Imports
import warnings
from datetime import datetime

from energis.run import rolling_horizon as rh

warnings.filterwarnings('ignore')

print("✅ Imports erfolgreich")

## 2. Konfiguration & Optimierung

In [ ]:
# Konfigurationsdateien
CONFIG_PATHS = [
    'configs/base.yaml',
    'configs/tech_catalog.yaml',
    'configs/sites/default.site.yaml',
    'configs/systems/baseline.system.yaml',
    'configs/scenarios/pf_then_rh.workflow.scenario.yaml',
]

print("📋 Konfigurationsdateien:")
for cfg_path in CONFIG_PATHS:
    full_path = PROJECT_ROOT / cfg_path
    exists = full_path.exists()
    symbol = '✅' if exists else '❌'
    print(f"  {symbol} {cfg_path}")

print("\n✅ Konfiguration OK")

In [ ]:
%%time
# Workflow ausführen
print("="*70)
print("🚀 STARTE OPTIMIERUNG")
print("="*70)
print(f"Start: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

try:
    workflow = rh.run_workflow(CONFIG_PATHS)
    
    print("\n" + "="*70)
    print("✅ OPTIMIERUNG ERFOLGREICH")
    print("="*70)
    print(f"\n📊 Workflow: {' → '.join(workflow.plan.steps)}")
    
    optimization_success = True
    
except Exception as e:
    print("\n" + "="*70)
    print("❌ FEHLER")
    print("="*70)
    print(f"\nFehler: {e}\n")
    
    import traceback
    traceback.print_exc()
    
    workflow = None
    optimization_success = False

## 3. Dashboard erstellen

Das Dashboard wird jetzt erstellt und kann interaktiv verwendet werden!

In [ ]:
if optimization_success and workflow:
    from energis.io.dashboard import create_dashboard, HAVE_PANEL, HAVE_PLOTLY
    
    if not HAVE_PANEL:
        print("❌ Panel nicht verfügbar!")
        print("   Installation: pip install panel holoviews bokeh")
    elif not HAVE_PLOTLY:
        print("⚠️  Plotly nicht verfügbar - Dashboard funktioniert mit eingeschränkten Features")
        print("   Installation: pip install plotly")
        dashboard = create_dashboard(workflow)
        dashboard
    else:
        print("🎛️ Dashboard wird erstellt...\n")
        
        # Dashboard erstellen
        dashboard = create_dashboard(
            workflow,
            title="EnerGIS Interactive Dashboard 🔥"
        )
        
        print("✅ Dashboard erfolgreich erstellt!")
        print("\n💡 Hinweise:")
        print("   • Navigiere zwischen Tabs mit den Reitern oben")
        print("   • Im Zeitreihen-Tab: Wähle Komponenten und Zeitbereich")
        print("   • Plots sind interaktiv: Zoom, Pan, Hover")
        print("   • In der Kostentabelle: Sortierung durch Klick auf Spalten")
        print("\n🌐 Als Webapp exportieren:")
        print("   panel serve interactive_dashboard.ipynb --show")
        
        # Dashboard anzeigen
        dashboard
else:
    print("⚠️  Keine Ergebnisse verfügbar - Dashboard kann nicht erstellt werden")

## 4. Export & Sharing

Das Dashboard kann als eigenständige Webapp exportiert werden.

In [ ]:
if optimization_success and workflow:
    print("📤 Export-Optionen:\n")
    
    print("1. Als HTML speichern (statisch):")
    print("   dashboard.save('dashboard.html')")
    print("")
    
    print("2. Als interaktive Webapp (benötigt Panel Server):")
    print("   panel serve interactive_dashboard.ipynb --show")
    print("   Oder: panel serve interactive_dashboard.ipynb --port 5006")
    print("")
    
    print("3. Dashboard-Komponente für andere Notebooks:")
    print("   from energis.io.dashboard import create_dashboard")
    print("   dashboard = create_dashboard(workflow)")
    print("   dashboard.servable()  # In Webapp-Modus")

## 5. Erweiterte Verwendung

Für fortgeschrittene Nutzer: Einzelne Dashboard-Komponenten können auch separat verwendet werden.

In [ ]:
# Beispiel: Nur Zeitreihen-Tab anzeigen
if optimization_success and workflow and HAVE_PANEL:
    from energis.io.dashboard import EnerGISDashboard
    
    dash_obj = EnerGISDashboard(workflow, "Custom View")
    
    # Zeige nur bestimmte Komponenten
    # timeseries_tab = dash_obj._create_timeseries_tab()
    # timeseries_tab
    
    print("💡 Tipp: Du kannst einzelne Tab-Funktionen aufrufen:")
    print("   - _create_overview_tab()")
    print("   - _create_timeseries_tab()")
    print("   - _create_costs_tab()")
    print("   - _create_design_tab()")
    print("   - _create_comparison_tab()")

---

## 🎓 Weitere Ressourcen

- **Panel Dokumentation**: https://panel.holoviz.org/
- **Plotly Dokumentation**: https://plotly.com/python/
- **EnerGIS README**: `../README.md`

## 🐛 Troubleshooting

**Dashboard wird nicht angezeigt:**
- Stelle sicher, dass Panel und Plotly installiert sind
- In JupyterLab: `jupyter labextension install @pyviz/jupyterlab_pyviz`
- Neustart des Kernels kann helfen

**Plots sind leer:**
- Überprüfe ob die Optimierung erfolgreich war
- Schaue ob Daten in workflow.pf_result oder workflow.rh_result vorhanden sind

**Webapp startet nicht:**
- Prüfe ob Port 5006 frei ist
- Versuche: `panel serve interactive_dashboard.ipynb --port 5007`

---

**Viel Erfolg mit dem Dashboard! 🚀**